# Vea: Visual Evidence AugmentationA walkthrough of one sample, stage by stage:1. **Attribution** - one forward pass, then read off which image patches the   visually-grounded layers attend to.2. **Evidence map** - denoise, smooth, normalize (Eqs. 2-3).3. **Highlighting** - dim everything that is not evidence (Eq. 4).4. **Effect** - compare the answer before and after.Requires a GPU and a downloaded checkpoint. Everything except the two model cellsruns on CPU.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from vea import PROMPTS, Attributor, VeaConfig, load_model, load_samples
from vea.attention import section_attention_stats
from vea.viz import show_attention_overlay, show_image, show_layer_grid

## 1. Load a sampleThe bundled TextVQA subset carries human-annotated evidence boxes, which is whatmakes the attribution measurable rather than merely visual.

In [ ]:
samples = load_samples("textvqa")
sample = samples[3]

print(f"{len(samples)} samples loaded")
print(f"Q: {sample.question}")
print(f"A: {sample.answers[0]}")
print(f"{len(sample.boxes)} evidence box(es): {sample.boxes}")

show_image(sample.load_image(), boxes=sample.boxes, title="original + annotated evidence")
plt.show()

## 2. Load a model`eager_attention=True` is essential: fused attention kernels never materialise theattention matrix, so `output_attentions=True` would silently return nothing.

In [ ]:
config = VeaConfig()          # paper defaults: alpha=0.5, sigma=0.5, lam=10
model = load_model("qwen2.5-vl-7b", vea_config=config)
print(f"{model.spec.hf_id}: {model.n_layers} layers on {model.device}")

## 3. AttributionOne forward pass gives the attention the final prompt token pays to every othertoken. That token is the one that produces the first answer token, so itsattention row is what actually drives the answer.

In [ ]:
attributor = Attributor(model, layers=None, config=config)   # layers=None -> all layers
attribution = attributor.attend(sample, PROMPTS["qa"])

layout = attribution.layout
print(f"prompt length      : {layout.n_input_tokens}")
print(f"image tokens       : {layout.image_span} ({layout.n_image_tokens} tokens)")
print(f"question tokens    : {layout.question_span}")
print(f"patch grid         : {layout.grid_size} of {layout.patch_size} px")
print(f"evidence patches   : {attribution.labels.n_evidence_patches} "
      f"({attribution.labels.evidence_ratio:.1%} of the image)")

## 4. Where does attention go, layer by layer?Shallow layers look nearly uniform over the image, middle layers unstructured, anddeep layers sparse but concentrated on the annotated region (red boxes).

In [ ]:
step = max(1, attribution.n_layers // 12)
show_layer_grid(
    attribution.input_image,
    attribution.attention,
    layout,
    layers=list(range(0, attribution.n_layers, step)),
    boxes=attribution.labels.boxes,
)
plt.show()

### Quantified: attention on evidence vs. non-evidence tokensRAPT (Relative Attention Per Token) divides a section's mean attention by theprompt-wide mean, which makes layers comparable despite differing scales. A valueabove 1 means the section gets more than its proportional share.

In [ ]:
stats = section_attention_stats(attribution.attention, layout, attribution.labels.patch_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
layers = np.arange(attribution.n_layers)

axes[0].plot(layers, stats["image_mean_norm"], label="image", marker="o", ms=3)
axes[0].plot(layers, stats["quest_mean_norm"], label="question", marker="s", ms=3)
axes[0].axhline(1.0, color="gray", ls=":", lw=1)
axes[0].set(xlabel="layer", ylabel="RAPT", title="text dominates early, image later")
axes[0].legend()

axes[1].plot(layers, stats["image_evd_mean_norm"], label="evidence", marker="o", ms=3)
axes[1].plot(layers, stats["image_nonevd_mean_norm"], label="non-evidence", marker="s", ms=3)
axes[1].set(xlabel="layer", ylabel="RAPT", title="deep layers single out the evidence")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Which layers are actually grounded?Scoring each layer by how well its attention *ranks* evidence patches (AUROC)identifies the visually-grounded layers. On a real run use`scripts/profile_layers.py` over ~100 samples; here we profile on a handful justto see the shape of the curve.

In [ ]:
from vea.profiling import profile_layers

profile = profile_layers(attributor, samples[:20], config=config)

plt.figure(figsize=(9, 3.2))
plt.bar(range(profile.n_layers), np.array(profile.layer_auroc) * 100, color="lightsteelblue")
plt.bar(profile.layers, np.array(profile.layer_auroc)[profile.layers] * 100,
        color="crimson", label="selected")
plt.axhline(50, color="gray", ls=":", lw=1, label="chance")
plt.xlabel("layer"); plt.ylabel("AUROC (%)"); plt.legend()
plt.title(f"{profile.model}: selected layers {profile.layers}")
plt.tight_layout(); plt.show()

print(f"all layers      : {profile.mean_auroc_all * 100:.2f} AUROC")
print(f"selected layers : {profile.mean_auroc_selected * 100:.2f} AUROC")

## 6. Build the evidence map (Eqs. 1-3)Each stage matters: denoising drops isolated encoder artifacts, smoothing removesthe blocky patch borders that would otherwise look like an unnatural mosaic.

In [ ]:
grounded = Attributor(model, layers=profile.layers, config=config)

variants = {
    "raw (no denoise, no smooth)": dict(denoise=False, smooth=False),
    "denoised":                    dict(denoise=True,  smooth=False),
    "denoised + smoothed (Vea)":   dict(denoise=True,  smooth=True),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, kwargs) in zip(axes, variants.items()):
    evidence_map = grounded.evidence_map(attribution, **kwargs)
    show_attention_overlay(attribution.input_image, evidence_map, title=name, ax=ax)
    from vea.viz import draw_boxes
    draw_boxes(ax, attribution.labels.boxes, color="lime")
plt.tight_layout(); plt.show()

## 7. Highlight and re-ask (Eq. 4)`alpha` is the brightness floor: evidence pixels keep their value, everything elseis scaled down toward `alpha`.

In [ ]:
augmented = grounded.augment(attribution)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
show_image(attribution.input_image, title="what the model normally sees", ax=axes[0])
show_image(augmented, title=f"Vea augmented (alpha={config.alpha})", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
from vea import evaluate_sample
from vea.config import resolve_method

for name in ["base", "inst", "vea"]:
    row = evaluate_sample(
        sample, resolve_method(name), model, grounded, PROMPTS, attribution=attribution
    )
    print(f"{name:<6} EM={row['em']:.0f}  F1={row['f1']:.2f}  -> {row['model_answer']!r}")

print(f"\ntrue answer: {sample.answers[0]!r}")

`base` vs `inst` isolates the prompt change; `inst` vs `vea` isolates the visualchange. If `vea` wins while `inst` does not, the gain came from the image.## 8. Scaling up```bashpython scripts/profile_layers.py  --model qwen2.5-vl-7b --limit 100python scripts/run_qa.py          --model qwen2.5-vl-7b --methods base inst vea \                                  --profile results/profiles/qwen2.5-vl-7b.jsonpython scripts/run_analysis.py    --model qwen2.5-vl-7bpython scripts/summarize.py qa```